### Dynamic Pricing using Cancellation Risk

In [ ]:
# Import libraries
import pandas as pd

# For ML model
import joblib

# Loading the data into a Pandas dataframe
hotel = pd.read_csv("../data/INNHotelsGroup.csv")


In [2]:
hotel.head()

,lead_time,market_segment_type,no_of_special_requests,avg_price_per_room,no_of_adults,no_of_weekend_nights,arrival_date,required_car_parking_space,no_of_week_nights,booking_status
0,224,Offline,0,65.00,2,1,2017-10-02,0,2,Not_Canceled
1,5,Online,1,106.68,2,2,2018-11-06,0,3,Not_Canceled
2,1,Online,0,60.00,1,2,2018-02-28,0,1,Canceled
3,211,Online,0,100.00,2,0,2018-05-20,0,2,Canceled
4,48,Online,0,94.50,2,1,2018-04-11,0,1,Canceled


#### More Preprocessing techniques 

In [3]:
# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

hotel['market_segment_type'] = LabelEncoder().fit_transform(hotel['market_segment_type'])
hotel['arrival_date'] = pd.to_datetime(hotel['arrival_date'])
hotel['arrival_month'] = hotel['arrival_date'].dt.month

# Select features for prediction
features = [
    'lead_time', 'market_segment_type', 'no_of_special_requests',
    'avg_price_per_room', 'no_of_adults', 'no_of_weekend_nights',
    'required_car_parking_space', 'no_of_week_nights', 'arrival_month'
]
X = hotel[features]


In [5]:
X.head()

,lead_time,market_segment_type,no_of_special_requests,avg_price_per_room,no_of_adults,no_of_weekend_nights,required_car_parking_space,no_of_week_nights,arrival_month
0,224,0,0,65.00,2,1,0,2,10
1,5,1,1,106.68,2,2,0,3,11
2,1,1,0,60.00,1,2,0,1,2
3,211,1,0,100.00,2,0,0,2,5
4,48,1,0,94.50,2,1,0,1,4


In [6]:
# Load trained cancellation prediction model
model = joblib.load('hotel_cancellation_prediction_model_v1_0.joblib')

# Get cancellation probabilities
cancel_probabilities = model.predict_proba(X)[:, 1]  # probability of cancellation (class 1)
hotel['cancellation_risk'] = cancel_probabilities


In [7]:
hotel['cancellation_risk'].head()

0    0.000000
1    0.085394
2    1.000000
3    0.979583
4    0.901850
Name: cancellation_risk, dtype: float64

In [8]:
def dynamic_pricing(base_price, cancellation_probability):
    # Pricing rule:
    if cancellation_probability >= 0.7:
        return base_price * 0.90  # 10% discount for high risk
    elif cancellation_probability >= 0.4:
        return base_price * 0.95  # 5% discount for moderate risk
    else:
        return base_price * 1.05  # 5% premium for low risk

# Apply dynamic pricing to each booking
hotel['dynamic_price'] = [
    dynamic_pricing(row['avg_price_per_room'], row['cancellation_risk'])
    for idx, row in hotel.iterrows()
]


In [9]:
# Show bookings with room rates adjusted based on predicted risk
output_cols = [
    'avg_price_per_room', 'cancellation_risk', 'dynamic_price'
]
print(hotel[output_cols].head(10))


   avg_price_per_room  cancellation_risk  dynamic_price
0               65.00           0.000000        68.2500
1              106.68           0.085394       112.0140
2               60.00           1.000000        54.0000
3              100.00           0.979583        90.0000
4               94.50           0.901850        85.0500
5              115.00           1.000000       103.5000
6              107.55           0.314422       112.9275
7              105.61           0.333250       110.8905
8               96.90           0.190859       101.7450
9              133.44           0.000000       140.1120


In [ ]:
# Save new pricing back to system or for further actions
hotel.to_csv('../data/INNHotelsGroup_dynamic_pricing.csv', index=False)
